# Move 4: RAG Evaluation Harness & Error Analysis

This notebook demonstrates the quantitative and qualitative evaluation framework of the **Production RAG Service**:
1. **Evaluation Architecture**: 15 test cases covering in-domain factual retrieval, multi-chunk synthesis, out-of-scope refusals, and adversarial prompt-injection queries.
2. **Evaluation Metrics**: Measuring `Recall@k`, `MRR`, `Faithfulness`, `Answer Relevancy`, `Context Precision`, `Context Recall`, and `Guardrail Accuracy`.
3. **Running the Harness Programmatically**: Executing `evals/run_eval.py` in offline mock mode or live with Gemini.
4. **Error & Refusal Analysis**: Deep dive into guardrail triggers, out-of-scope evidence cutoffs, and latency.

In [ ]:
import json
import pandas as pd
from evals.run_eval import run_evaluation

print("Evaluation modules loaded successfully!")

## 1. Inspecting the Golden Evaluation Dataset

In [ ]:
with open("evals/golden_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

df_cases = pd.DataFrame(dataset)
print(f"Total Test Cases: {len(df_cases)}")
print("\nTest Case Distribution by Category:")
print(df_cases["category"].value_axis() if hasattr(df_cases["category"], 'value_axis') else df_cases["category"].value_counts())
df_cases[["id", "category", "query", "expected_status", "expected_refused"]].head(10)

## 2. Running Evaluation Programmatically (Offline Mode)

In [ ]:
# Execute evaluation across all 15 cases in offline mode
summary = await run_evaluation(
    dataset_path="evals/golden_dataset.json",
    output_dir="evals/results",
    offline=True,
    sleep_seconds=0.0,
    user_id="eval_analysis_user",
)

print("\n--- EVALUATION SUMMARY METRICS ---")
print(f"Retrieval Recall@k:              {summary['retrieval']['recall_at_k'] * 100:.1f}%")
print(f"Mean Reciprocal Rank (MRR):       {summary['retrieval']['mrr']:.3f}")
print(f"Prompt Injection Block Rate:      {summary['guardrails']['prompt_injection_block_rate'] * 100:.1f}%")
print(f"Out-of-Scope Refusal Rate:        {summary['guardrails']['out_of_scope_refusal_rate'] * 100:.1f}%")
print(f"Faithfulness:                     {summary['generation_quality']['faithfulness']:.3f}")
print(f"Answer Relevancy:                 {summary['generation_quality']['answer_relevancy']:.3f}")
print(f"Average Latency:                  {summary['performance']['avg_latency_ms']:.2f} ms")

## 3. Case-by-Case Breakdown & Error Analysis

In [ ]:
df_results = pd.DataFrame(summary["details"])
print("=== Performance by Test Case ===")
df_results[["id", "category", "query", "pass", "latency_ms"]]

## 4. Security Guardrail Verification

Verify that 100% of malicious prompt injections were intercepted and returned `400 Bad Request`.

In [ ]:
injections = df_results[df_results["category"] == "prompt_injection"]
all_blocked = injections["blocked"].all()
print(f"All Adversarial Injections Blocked (HTTP 400): {all_blocked}")
injections[["id", "query", "status_code", "blocked", "pass"]]